In [3]:
#загрузка списка топ 200 по капитализации

from __future__ import annotations
import os
import re
import html
import time
import tempfile
from typing import List

import requests
import pandas as pd
from bs4 import BeautifulSoup


try:
    import certifi, shutil
    caf = certifi.where()
    if any(ord(ch) > 127 for ch in caf):
        ascii_dir = os.path.join(os.environ.get("TEMP", tempfile.gettempdir()), "certifi_ascii")
        os.makedirs(ascii_dir, exist_ok=True)
        ascii_caf = os.path.join(ascii_dir, "cacert.pem")
        if not os.path.exists(ascii_caf):
            shutil.copyfile(caf, ascii_caf)
        caf = ascii_caf
    os.environ["SSL_CERT_FILE"] = caf
    os.environ["REQUESTS_CA_BUNDLE"] = caf
    os.environ["CURL_CA_BUNDLE"] = caf
    os.environ["YF_USE_CURL_CFFI"] = "0"
except Exception:
    pass

import yfinance as yf


BASE_URL = "https://companiesmarketcap.com/usa/largest-companies-in-the-usa-by-market-cap/"
HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; us-top200-downloader/1.0)"}

OUT_DIR = "data"
START_DATE = "2008-01-01"
END_DATE = None
INTERVAL = "1d"
TOP_N = 200

BATCH_SIZE = 50
MAX_RETRIES = 3
RETRY_SLEEP = 2.0


def _get_html(url: str) -> str:
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            r = requests.get(url, headers=HEADERS, timeout=30)
            r.raise_for_status()
            return r.text
        except Exception:
            if attempt == MAX_RETRIES:
                raise
            time.sleep(RETRY_SLEEP)

def _extract_tickers_from_html(html_text: str) -> list[str]:

    soup = BeautifulSoup(html_text, "html.parser")
    anchors = soup.select("a")


    def looks_like_company_href(href: str) -> bool:
        if not href:
            return False
        href = href.lower()
        return (
            "/company/" in href
            or "/marketcap/" in href
            or ("usa" in href and "companies" in href)
        )

    pat = re.compile(r"\b([A-Z]{1,5}(?:-[A-Z])?)\b$")

    tickers: list[str] = []
    seen = set()

    for a in anchors:
        href = a.get("href") or ""
        if not looks_like_company_href(href):
            continue
        text = " ".join(a.stripped_strings)
        if not text:
            continue
        text = html.unescape(text)
        m = pat.search(text)
        if not m:
            continue
        t = m.group(1).upper().replace(".", "-")
        if not (1 <= len(t) <= 6):
            continue
        if t in {"ETF", "USA"}:
            continue
        if t not in seen:
            seen.add(t)
            tickers.append(t)

    return tickers

def get_us_top_n_tickers(n: int = 200) -> list[str]:
    """
    Собираем тикеры с первых страниц рейтинга (1..3 для надёжности), пока не наберём n.
    """
    all_tickers: list[str] = []
    for page in (1, 2, 3):
        url = BASE_URL if page == 1 else f"{BASE_URL}?page={page}"
        html_text = _get_html(url)
        page_tickers = _extract_tickers_from_html(html_text)

        for t in page_tickers:
            if t not in all_tickers:
                all_tickers.append(t)
        if len(all_tickers) >= n:
            break

    return all_tickers[:n]

def _chunked(seq: List[str], size: int) -> List[List[str]]:
    return [seq[i:i+size] for i in range(0, len(seq), size)]

def download_prices(tickers: List[str]) -> pd.DataFrame:
    """
    Скачивает котировки батчами и возвращает tidy-таблицу:
    столбцы: date, Ticker, Open, High, Low, Close, Adj Close, Volume
    """
    all_batches = []
    for i, batch in enumerate(_chunked(tickers, BATCH_SIZE), start=1):
        print(f"[{i}] Download: {', '.join(batch[:8])}{' ...' if len(batch)>8 else ''}")
        for attempt in range(1, MAX_RETRIES + 1):
            try:
                df = yf.download(
                    tickers=batch,
                    start=START_DATE,
                    end=END_DATE,
                    interval=INTERVAL,
                    auto_adjust=True,
                    group_by="ticker",
                    threads=True,
                    progress=False,
                )
                if isinstance(df, pd.DataFrame) and not df.empty:
                    all_batches.append(df)
                    break
                raise RuntimeError("Empty frame returned")
            except Exception as e:
                if attempt == MAX_RETRIES:
                    print(f"  !! failed batch {i}: {e}")
                else:
                    time.sleep(RETRY_SLEEP)
        time.sleep(0.5)

    if not all_batches:
        raise SystemExit("Не удалось скачать котировки — все батчи пустые.")

    wide = pd.concat(all_batches, axis=1)

    if isinstance(wide.columns, pd.MultiIndex):
        top = list(wide.columns.levels[0])
        if any(k in {"Open", "Close", "Adj Close", "Volume"} for k in top):
            wide = wide.swaplevel(axis=1).sort_index(axis=1)
    else:
        wide = pd.concat({"SINGLE": wide}, axis=1)

    wide.columns = pd.MultiIndex.from_tuples(
        [(str(tick), str(field)) for (tick, field) in wide.columns],
        names=["Ticker", "Field"]
    )
    long_df = wide.stack(level=["Ticker"]).reset_index().rename(columns={"Date": "date"})
    if "level_0" in long_df.columns:
        long_df = long_df.rename(columns={"level_0": "date"})

    possible = ["Open","High","Low","Close","Adj Close","Volume"]
    cols = ["date","Ticker"] + [c for c in possible if c in long_df.columns]
    if all(f not in long_df.columns for f in possible) and "Field" in long_df.columns:
        value_col = next((c for c in long_df.columns if c not in {"date","Ticker","Field"}), None)
        out = long_df.pivot_table(index=["date","Ticker"], columns="Field", values=value_col, aggfunc="first").reset_index()
        out.columns.name = None
        order = ["date","Ticker"] + [c for c in possible if c in out.columns] + \
                [c for c in out.columns if c not in {"date","Ticker", *possible}]
        tidy = out[order].sort_values(["Ticker","date"])
    else:
        tidy = long_df[cols].sort_values(["Ticker","date"])

    return tidy

def save_csvs(tidy: pd.DataFrame, requested_tickers: list[str]) -> None:
    os.makedirs(OUT_DIR, exist_ok=True)
    tidy.to_csv(os.path.join(OUT_DIR, "prices_all.csv"), index=False)

    present = set(tidy["Ticker"].unique())
    missing = [t for t in requested_tickers if t not in present]

    for t in sorted(present):
        df_t = tidy[tidy["Ticker"] == t].reset_index(drop=True)
        df_t.to_csv(os.path.join(OUT_DIR, f"{t}.csv"), index=False)

    with open(os.path.join(OUT_DIR, "top200_tickers.csv"), "w", encoding="utf-8") as f:
        f.write("Ticker\n")
        for t in requested_tickers:
            f.write(f"{t}\n")
    if missing:
        with open(os.path.join(OUT_DIR, "failed_tickers.txt"), "w", encoding="utf-8") as f:
            f.write("\n".join(missing))
        print(f"⚠ Не скачались {len(missing)} тикеров. Список: data/failed_tickers.txt")

def main():
    print("Получаем топ-200 компаний США по капитализации…")
    tickers = get_us_top_n_tickers(TOP_N)
    print(f"Найдено тикеров: {len(tickers)}")
    if len(tickers) < 150:
        print("Предупреждение: извлечено мало тикеров — возможно, изменилась разметка сайта.")

    tidy = download_prices(tickers)
    save_csvs(tidy, tickers)
    print(f"Готово. CSV в ./{OUT_DIR}/")

if __name__ == "__main__":
    main()


Получаем топ-200 компаний США по капитализации…
Найдено тикеров: 200
[1] Download: NVDA, AAPL, MSFT, GOOG, AMZN, META, AVGO, TSLA ...
[2] Download: ANET, NOW, LRCX, INTU, BX, QCOM, AMAT, INTC ...
[3] Download: VRTX, DELL, SO, SCCO, HCA, CVS, NKE, MCK ...
[4] Download: AJG, ITW, APO, VRT, WMB, RSG, VST, MNST ...


C:\Users\allll\AppData\Local\Temp\ipykernel_1120\858605971.py:181: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  long_df = wide.stack(level=["Ticker"]).reset_index().rename(columns={"Date": "date"})


Готово. CSV в ./data/


In [4]:
# выгрузка данных по топ 200

from __future__ import annotations
import os
import glob
import pandas as pd
from datetime import datetime
from typing import List, Dict

from dash import Dash, dcc, html, Input, Output, State, no_update
import plotly.graph_objs as go

DATA_DIR = "data"
ALL_CSV = os.path.join(DATA_DIR, "prices_all.csv")

def load_data() -> pd.DataFrame:
    """
    Пытается прочитать data/prices_all.csv.
    Если его нет — читает все CSV из data/*.csv, где есть колонка Ticker.
    Возвращает tidy-таблицу со столбцами минимум: date, Ticker, Close (и др. если есть).
    """
    if os.path.exists(ALL_CSV):
        df = pd.read_csv(ALL_CSV)
    else:
        parts: List[pd.DataFrame] = []
        for path in glob.glob(os.path.join(DATA_DIR, "*.csv")):
            try:
                dfi = pd.read_csv(path)
                if "Ticker" in dfi.columns:
                    parts.append(dfi)
            except Exception:
                pass
        if not parts:
            raise SystemExit("Не найден ни один CSV с колонкой 'Ticker' в ./data/")
        df = pd.concat(parts, ignore_index=True)

    colmap = {c.lower().strip(): c for c in df.columns}
    def col_get(name: str) -> str | None:
        return colmap.get(name.lower())

    ren = {}
    for want in ["date","Ticker","Open","High","Low","Close","Adj Close","Volume"]:
        have = None
        for c in df.columns:
            if c.strip().lower() == want.lower():
                have = c
                break
        if have and have != want:
            ren[have] = want
    if ren:
        df = df.rename(columns=ren)

    if "date" not in df.columns or "Ticker" not in df.columns:
        raise ValueError("В CSV должны быть как минимум колонки 'date' и 'Ticker'.")

    df["date"] = pd.to_datetime(df["date"])

    cols_keep = ["date","Ticker","Open","High","Low","Close","Adj Close","Volume"]
    present = [c for c in cols_keep if c in df.columns]
    df = df[present].sort_values(["Ticker","date"]).reset_index(drop=True)

    return df

df = load_data()
TICKERS = sorted(df["Ticker"].unique())
DATE_MIN = df["date"].min().date()
DATE_MAX = df["date"].max().date()
PRICE_FIELDS = [c for c in ["Adj Close","Close","Open","High","Low"] if c in df.columns]


app = Dash(__name__)
app.title = "US Stocks Interactive"

app.layout = html.Div([
    html.H2("US Stocks — интерактивные графики"),
    html.Div([
        html.Div([
            html.Label("Компания (Ticker)"),
            dcc.Dropdown(
                id="ticker-dd",
                options=[{"label": t, "value": t} for t in TICKERS],
                value=TICKERS[0],
                clearable=False,
                style={"minWidth":"220px"}
            ),
        ], style={"display":"inline-block","marginRight":"16px","verticalAlign":"top"}),

        html.Div([
            html.Label("Показатель цены"),
            dcc.Dropdown(
                id="price-field-dd",
                options=[{"label": f, "value": f} for f in PRICE_FIELDS],
                value=PRICE_FIELDS[0],
                clearable=False,
                style={"minWidth":"180px"}
            ),
        ], style={"display":"inline-block","marginRight":"16px","verticalAlign":"top"}),

        html.Div([
            html.Label("Период дат"),
            dcc.DatePickerRange(
                id="date-range",
                min_date_allowed=DATE_MIN,
                max_date_allowed=DATE_MAX,
                start_date=max(DATE_MIN, (DATE_MAX.replace(year=DATE_MAX.year-1))),  # последний год по умолчанию
                end_date=DATE_MAX,
                display_format="YYYY-MM-DD"
            ),
        ], style={"display":"inline-block","marginRight":"16px","verticalAlign":"top"}),

        html.Div([
            html.Label("Скользящая средняя (дней)"),
            dcc.Input(id="ma-window", type="number", value=20, min=1, step=1, style={"width":"90px"})
        ], style={"display":"inline-block","marginRight":"16px","verticalAlign":"top"}),

    ], style={"marginBottom":"12px"}),

    dcc.Graph(id="price-graph", style={"height":"72vh"}),

    html.Hr(),
    html.Div(id="hover-info", style={"whiteSpace":"pre-wrap","fontFamily":"monospace"})
], style={"padding":"16px"})


def make_figure(sub: pd.DataFrame, price_field: str) -> go.Figure:
    def hovertext(row) -> str:
        parts = [f"Date: {row['date'].date()}"]
        for col in ["Open","High","Low","Close","Adj Close","Volume"]:
            if col in sub.columns:
                val = row[col]
                if col == "Volume" and pd.notna(val):
                    parts.append(f"{col}: {int(val):,}".replace(",", " "))
                else:
                    parts.append(f"{col}: {val}")
        return "<br>".join(parts)

    ht = sub.apply(hovertext, axis=1)

    fig = go.Figure()

    # Линия цены
    fig.add_trace(go.Scatter(
        x=sub["date"],
        y=sub[price_field],
        mode="lines",
        name=price_field,
        hoverinfo="text",
        hovertext=ht
    ))

    # MA
    if "MA" in sub.columns:
        fig.add_trace(go.Scatter(
            x=sub["date"], y=sub["MA"], mode="lines",
            name=f"MA({sub.attrs.get('ma_window', 'N')})",
            line=dict(dash="dot"),
            hoverinfo="skip"
        ))

    # Объём
    if "Volume" in sub.columns:
        fig.add_trace(go.Bar(
            x=sub["date"],
            y=sub["Volume"],
            name="Volume",
            yaxis="y2",
            opacity=0.3,
            hoverinfo="skip"
        ))

    fig.update_layout(
        margin=dict(l=40, r=40, t=50, b=40),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
        hovermode="x unified",
        xaxis=dict(title=None, showgrid=True),
        yaxis=dict(title=price_field, showgrid=True),
        yaxis2=dict(title="Volume", overlaying="y", side="right", showgrid=False),
    )
    return fig


@app.callback(
    Output("price-graph", "figure"),
    Output("hover-info", "children"),
    Input("ticker-dd", "value"),
    Input("price-field-dd", "value"),
    Input("date-range", "start_date"),
    Input("date-range", "end_date"),
    Input("ma-window", "value"),
)
def update_graph(ticker: str, price_field: str, start_date: str, end_date: str, ma_window: int):
    if not ticker or not price_field:
        return no_update, ""

    # фильтрация
    mask = (df["Ticker"] == ticker)
    if start_date:
        mask &= (df["date"] >= pd.to_datetime(start_date))
    if end_date:
        mask &= (df["date"] <= pd.to_datetime(end_date))

    sub = df.loc[mask].sort_values("date").copy()
    if sub.empty:
        fig = go.Figure()
        fig.update_layout(title="Нет данных для выбранного фильтра")
        return fig, ""

    # скользящая средняя
    if isinstance(ma_window, (int, float)) and ma_window and ma_window > 1 and price_field in sub.columns:
        sub["MA"] = sub[price_field].rolling(int(ma_window), min_periods=1).mean()
        sub.attrs["ma_window"] = int(ma_window)

    fig = make_figure(sub, price_field)

    # краткая сводка периода
    info_lines = []
    info_lines.append(f"Ticker: {ticker}")
    info_lines.append(f"Period: {sub['date'].iloc[0].date()} — {sub['date'].iloc[-1].date()}")
    if price_field in sub.columns:
        info_lines.append(f"{price_field}: first={sub[price_field].iloc[0]} last={sub[price_field].iloc[-1]}")
    if "Volume" in sub.columns:
        total_vol = int(sub["Volume"].sum())
        info_lines.append(f"Total Volume: {total_vol:,}".replace(",", " "))
    return fig, "\n".join(info_lines)


if __name__ == "__main__":
    # host="0.0.0.0" — если хотите открыть с другого устройства в сети
    app.run(debug=True)


In [5]:
# поиск аномалий в данных

from __future__ import annotations
import os
import glob
import argparse
import sys
import numpy as np
import pandas as pd

DEFAULT_DATA_DIR = "data"
DEFAULT_ALL_CSV = os.path.join(DEFAULT_DATA_DIR, "prices_all.csv")
QA_DIR = os.path.join(DEFAULT_DATA_DIR, "qa")

PRICE_COLS_ALL = ["Open","High","Low","Close","Adj Close"]
VOL_COL = "Volume"
REQUIRED_COLS = ["date","Ticker"]

def parse_args():
    ap = argparse.ArgumentParser(
        description="Проверка качества CSV котировок (пропуски, дубликаты, аномалии).",
        add_help=True
    )
    ap.add_argument("--data-dir", default=DEFAULT_DATA_DIR, help="Каталог с CSV (по умолчанию ./data)")
    ap.add_argument("--all-csv", default=DEFAULT_ALL_CSV, help="Путь к общему CSV (если нет — читаем все *.csv)")
    ap.add_argument("--ret-threshold", type=float, default=0.30,
                    help="Порог для |доходности| как аномалии (pct_change), по умолчанию 0.30 = ±30%")
    ap.add_argument("--vol-z", type=float, default=5.0,
                    help="Порог Z-скор для аномалий объёма (по умолчанию 5σ)")
    ap.add_argument("--max_gap_days", type=int, default=5,
                    help="Разрыв, если интервал между соседними датами > N дней (по умолчанию 5)")
    args, _unknown = ap.parse_known_args(sys.argv[1:])  # важно для Jupyter
    return args

def load_data(path_all_csv: str, data_dir: str) -> pd.DataFrame:
    if os.path.exists(path_all_csv):
        df = pd.read_csv(path_all_csv)
    else:
        parts = []
        for p in glob.glob(os.path.join(data_dir, "*.csv")):
            try:
                dfi = pd.read_csv(p)
                if "Ticker" in dfi.columns:
                    parts.append(dfi)
            except Exception:
                pass
        if not parts:
            raise SystemExit(f"Не найдено CSV с колонкой 'Ticker' в {data_dir}")
        df = pd.concat(parts, ignore_index=True)

    # нормализация названий
    ren = {}
    for want in REQUIRED_COLS + PRICE_COLS_ALL + [VOL_COL]:
        for c in df.columns:
            if c.strip().lower() == want.lower():
                ren[c] = want
                break
    df = df.rename(columns=ren)

    # базовые проверки
    for c in REQUIRED_COLS:
        if c not in df.columns:
            raise SystemExit(f"В данных отсутствует обязательная колонка: {c}")

    # типы
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    if df["date"].isna().any():
        raise SystemExit("Есть некорректные значения дат (не парсятся). Очистите данные.")

    # упорядочим и оставим только известные поля
    present_price_cols = [c for c in PRICE_COLS_ALL if c in df.columns]
    keep = ["date","Ticker"] + present_price_cols + ([VOL_COL] if VOL_COL in df.columns else [])
    df = df[keep].sort_values(["Ticker","date"]).reset_index(drop=True)
    return df

def detect_interval_days(dates: pd.Series) -> int:
    if dates.size < 3:
        return 1
    deltas = np.diff(dates.values.astype("datetime64[D]").astype("int64"))
    if deltas.size == 0:
        return 1
    vals, counts = np.unique(deltas, return_counts=True)
    return int(vals[counts.argmax()])

def main():
    args = parse_args()
    os.makedirs(QA_DIR, exist_ok=True)

    df = load_data(args.all_csv, args.data_dir)
    tickers = df["Ticker"].unique().tolist()

    # Определим, какие колонки реально есть
    price_present_cols = [c for c in PRICE_COLS_ALL if c in df.columns]
    has_volume = VOL_COL in df.columns
    cols_for_missing = price_present_cols + ([VOL_COL] if has_volume else [])

    # 1) Дубликаты по ключу (Ticker, date)
    dup_mask = df.duplicated(subset=["Ticker","date"], keep=False)
    dups = df.loc[dup_mask].copy()
    dups.to_csv(os.path.join(QA_DIR, "duplicates.csv"), index=False)

    # 2) Пропуски: общее и по тикерам
    miss_total = df.isna().sum().to_frame("missing_count")
    miss_total["missing_pct"] = (miss_total["missing_count"] / len(df)).round(6)
    miss_total.to_csv(os.path.join(QA_DIR, "missing_total.csv"))

    if cols_for_missing:
        miss_by_ticker = (df.groupby("Ticker")[cols_for_missing]
                            .apply(lambda g: g.isna().sum())
                            .reset_index())
        counts_by_ticker = df.groupby("Ticker").size().rename("rows").reset_index()
        miss_by_ticker = miss_by_ticker.merge(counts_by_ticker, on="Ticker", how="left")
        for c in cols_for_missing:
            miss_by_ticker[c+"_pct"] = (miss_by_ticker[c] / miss_by_ticker["rows"]).round(6)
        miss_by_ticker.to_csv(os.path.join(QA_DIR, "missing_by_ticker.csv"), index=False)
    else:
        pd.DataFrame(columns=["Ticker","rows"]).to_csv(os.path.join(QA_DIR, "missing_by_ticker.csv"), index=False)

    # 3) Невалидные значения и логические проверки OHLC
    bad_rows = []

    def add_bad(mask: pd.Series, reason: str):
        if mask.any():
            cols = ["date","Ticker"] + price_present_cols + ([VOL_COL] if has_volume else [])
            x = df.loc[mask, cols].copy()
            x.insert(2, "reason", reason)
            bad_rows.append(x)

    # отрицательные/нулевые цены
    for c in price_present_cols:
        add_bad(df[c] <= 0, f"{c} <= 0")

    # отрицательный объём
    if has_volume:
        add_bad(df[VOL_COL] < 0, "Volume < 0")

    # OHLC консистентность при наличии High/Low
    if "High" in price_present_cols and "Low" in price_present_cols:
        add_bad(df["High"] < df["Low"], "High < Low")
        for c in [x for x in ["Open","Close"] if x in price_present_cols]:
            add_bad(df[c] < df["Low"], f"{c} < Low")
            add_bad(df[c] > df["High"], f"{c} > High")

    bad_df = pd.concat(bad_rows, ignore_index=True) if bad_rows else pd.DataFrame(columns=["date","Ticker","reason"])
    bad_df.to_csv(os.path.join(QA_DIR, "anomalies_ohlc_rules.csv"), index=False)

    # 4) Аномальные доходности (Adj Close, иначе Close) и объёмы
    returns_rows = []
    vol_spike_rows = []
    ret_col = "Adj Close" if "Adj Close" in price_present_cols else ("Close" if "Close" in price_present_cols else None)

    for t, g in df.groupby("Ticker", sort=False):
        g = g.sort_values("date").copy()

        # Доходности
        if ret_col is not None and ret_col in g.columns:
            g["ret"] = g[ret_col].pct_change()
            mask = g["ret"].abs() > args.ret_threshold
            if mask.any():
                r = g.loc[mask, ["date","Ticker",ret_col,"ret"]].copy()
                r = r.rename(columns={ret_col: "price_col"})
                returns_rows.append(r)

        # Спайки объёма
        if has_volume:
            vv = g[VOL_COL].replace(0, np.nan).dropna()
            if vv.size >= 10:
                lv = np.log(vv)
                mu, sigma = lv.mean(), lv.std(ddof=1)
                if sigma > 0:
                    z = (np.log(g[VOL_COL].where(g[VOL_COL] > 0, np.nan)) - mu) / sigma
                    spike = z.abs() > args.vol_z
                    if spike.any():
                        v = g.loc[spike, ["date","Ticker",VOL_COL]].copy()
                        v["z"] = z.loc[spike].values
                        vol_spike_rows.append(v)

    ret_out = pd.concat(returns_rows, ignore_index=True) if returns_rows else pd.DataFrame(columns=["date","Ticker","price_col","ret"])
    vol_out = pd.concat(vol_spike_rows, ignore_index=True) if vol_spike_rows else pd.DataFrame(columns=["date","Ticker",VOL_COL,"z"])
    ret_out.to_csv(os.path.join(QA_DIR, "anomalies_returns.csv"), index=False)
    vol_out.to_csv(os.path.join(QA_DIR, "anomalies_volume_spikes.csv"), index=False)

    # 5) Нулевой объём и полностью пустые строки по ценам
    if has_volume:
        zero_vol = df.loc[df[VOL_COL] == 0, ["date","Ticker",VOL_COL]].copy()
        zero_vol.to_csv(os.path.join(QA_DIR, "zero_volume.csv"), index=False)
    else:
        zero_vol = pd.DataFrame()

    all_price_na = pd.DataFrame(columns=["date","Ticker"])
    if price_present_cols:
        tmp = df[["date","Ticker"] + price_present_cols].copy()
        row_all_na = tmp[price_present_cols].isna().all(axis=1)
        all_price_na = tmp.loc[row_all_na]
        all_price_na.to_csv(os.path.join(QA_DIR, "all_price_fields_missing.csv"), index=False)

    # 6) Разрывы дат (эвристика)
    gaps_rows = []
    for t, g in df.groupby("Ticker", sort=False):
        g = g.sort_values("date")
        step = detect_interval_days(g["date"])
        dd = g["date"].diff().dt.days
        thr = max(args.max_gap_days, 3*step)
        gap_mask = dd > thr
        if gap_mask.any():
            gg = pd.DataFrame({
                "Ticker": t,
                "prev_date": g["date"].shift(1).loc[gap_mask].values,
                "next_date": g["date"].loc[gap_mask].values,
                "delta_days": dd.loc[gap_mask].values
            })
            gaps_rows.append(gg)
    gaps_out = pd.concat(gaps_rows, ignore_index=True) if gaps_rows else pd.DataFrame(columns=["Ticker","prev_date","next_date","delta_days"])
    gaps_out.to_csv(os.path.join(QA_DIR, "date_gaps.csv"), index=False)

    # 7) Сводка по тикерам
    stats = (df.groupby("Ticker")
               .agg(first_date=("date","min"),
                    last_date=("date","max"),
                    rows=("date","size"))
               .reset_index())
    uniq_dates = df.groupby("Ticker")["date"].nunique().rename("unique_dates")
    stats = stats.merge(uniq_dates, on="Ticker", how="left")
    if price_present_cols:
        na_any = df.groupby("Ticker")[price_present_cols].apply(lambda g: g.isna().any(axis=1).mean()).rename("rows_with_any_na_pct")
        stats = stats.merge(na_any, on="Ticker", how="left")
    dup_cnt = dups.groupby("Ticker").size().rename("duplicate_rows").reset_index()
    stats = stats.merge(dup_cnt, on="Ticker", how="left")
    stats["duplicate_rows"] = stats["duplicate_rows"].fillna(0).astype(int)
    stats.to_csv(os.path.join(QA_DIR, "summary_by_ticker.csv"), index=False)

    # 8) Итого
    print("\n=== DATA QUALITY SUMMARY ===")
    print(f"Tickers: {len(tickers)}")
    print(f"Rows: {len(df)}")
    print(f"Duplicates: {len(dups)}  → {os.path.join(QA_DIR, 'duplicates.csv')}")
    print(f"Missing (total by column) → {os.path.join(QA_DIR, 'missing_total.csv')}")
    print(f"Missing by ticker → {os.path.join(QA_DIR, 'missing_by_ticker.csv')}")
    print(f"OHLC rule violations: {len(bad_df)} → {os.path.join(QA_DIR, 'anomalies_ohlc_rules.csv')}")
    print(f"Big return spikes: {len(ret_out)} → {os.path.join(QA_DIR, 'anomalies_returns.csv')} (>|{args.ret_threshold:.2f}|, col={ret_col})")
    print(f"Volume spikes: {len(vol_out)} → {os.path.join(QA_DIR, 'anomalies_volume_spikes.csv')} (> {args.vol_z:.1f}σ)")
    print(f"Zero volume rows: {len(zero_vol)} → {os.path.join(QA_DIR, 'zero_volume.csv')}")
    print(f"Date gaps: {len(gaps_out)} → {os.path.join(QA_DIR, 'date_gaps.csv')} (thr≥{max(args.max_gap_days,3)} days)")
    print(f"Per-ticker stats → {os.path.join(QA_DIR, 'summary_by_ticker.csv')}")
    print("Done.")

if __name__ == "__main__":
    main()



=== DATA QUALITY SUMMARY ===
Tickers: 200
Rows: 813832
Duplicates: 0  → data\qa\duplicates.csv
Missing (total by column) → data\qa\missing_total.csv
Missing by ticker → data\qa\missing_by_ticker.csv
OHLC rule violations: 331 → data\qa\anomalies_ohlc_rules.csv
Big return spikes: 65 → data\qa\anomalies_returns.csv (>|0.30|, col=Close)
Volume spikes: 216 → data\qa\anomalies_volume_spikes.csv (> 5.0σ)
Zero volume rows: 53 → data\qa\zero_volume.csv
Date gaps: 0 → data\qa\date_gaps.csv (thr≥5 days)
Per-ticker stats → data\qa\summary_by_ticker.csv
Done.


In [3]:
import os
import glob
import pandas as pd

DATA_DIR = "data"
ALL_CSV = os.path.join(DATA_DIR, "prices_all.csv")

if os.path.exists(ALL_CSV):
    df = pd.read_csv(ALL_CSV)
else:
    parts = []
    for p in glob.glob(os.path.join(DATA_DIR, "*.csv")):
        try:
            dfi = pd.read_csv(p)
            if "Ticker" in dfi.columns or "ticker" in dfi.columns:
                parts.append(dfi)
        except Exception:
            pass
    if not parts:
        raise SystemExit("Не нашёл данных в ./data/")
    df = pd.concat(parts, ignore_index=True)

ren = {}
for want in ["date","Ticker"]:
    for c in df.columns:
        if c.strip().lower() == want.lower():
            ren[c] = want
            break
df = df.rename(columns=ren)
if "date" not in df.columns or "Ticker" not in df.columns:
    raise SystemExit("Нужны колонки: date и Ticker")

df["date"] = pd.to_datetime(df["date"])

first_global = df["date"].min()
first_by_ticker = df.groupby("Ticker")["date"].min().reset_index(name="first_date")
tickers_from_start = first_by_ticker[first_by_ticker["first_date"] == first_global]["Ticker"].sort_values()

print(f"Самая ранняя дата в данных: {first_global.date()}")
print(f"Тикеров, начинающихся с этой даты: {len(tickers_from_start)}")
print(", ".join(tickers_from_start.tolist()))



Самая ранняя дата в данных: 2008-01-02
Тикеров, начинающихся с этой даты: 159
AAPL, ABT, ADBE, ADI, ADP, ADSK, AEP, AFL, AJG, ALNY, AMAT, AMD, AMGN, AMT, AMZN, APD, APH, AXON, AXP, AZO, BA, BAC, BK, BKNG, BLK, BMY, BRK-B, BSX, BX, C, CAT, CDNS, CI, CL, CMCSA, CME, CMG, CMI, COF, COP, COR, COST, CRM, CSCO, CSX, CTAS, CVS, CVX, DE, DHR, DIS, DLR, DUK, ECL, ELV, EMR, EOG, EPD, EQIX, ET, F, FCX, FDX, FI, GD, GE, GILD, GLW, GOOG, GS, HD, HON, IBKR, IBM, ICE, INTC, INTU, ISRG, ITW, JNJ, JPM, KLAC, KO, LHX, LLY, LMT, LOW, LRCX, MA, MAR, MCD, MCK, MCO, MDLZ, MMC, MMM, MNST, MO, MRK, MRVL, MS, MSFT, MSI, MSTR, MU, NEE, NEM, NFLX, NKE, NOC, NSC, NVDA, O, ORCL, ORLY, PEP, PFE, PG, PGR, PH, PLD, PNC, PWR, QCOM, RCL, REGN, RSG, RTX, SBUX, SCCO, SCHW, SHW, SNPS, SO, SPG, SPGI, SRE, SYK, T, TDG, TFC, TJX, TMO, TMUS, TRV, TXN, UNH, UNP, UPS, URI, USB, VRTX, VZ, WELL, WFC, WM, WMB, WMT, XOM


[INFO] Device: cuda
[INFO] CUDA build: 12.1
[INFO] GPU: NVIDIA GeForce RTX 2060
[INFO] Using price column: Close
[INFO] Global first date: 2008-01-02
[INFO] Tickers starting at first date: 159
[INFO] Matrix: companies=159, time_points=4483 (full period)


grid:   1%|▏         | 1/72 [00:05<06:22,  5.39s/it]C:\Users\allll\PycharmProjects\проект мага\.venv\Lib\site-packages\sklearn\base.py:1365: ConvergenceWarning: Number of distinct clusters (1) found smaller than n_clusters (6). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
C:\Users\allll\PycharmProjects\проект мага\.venv\Lib\site-packages\sklearn\base.py:1365: ConvergenceWarning: Number of distinct clusters (1) found smaller than n_clusters (8). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
C:\Users\allll\PycharmProjects\проект мага\.venv\Lib\site-packages\sklearn\base.py:1365: ConvergenceWarning: Number of distinct clusters (1) found smaller than n_clusters (10). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
grid:  26%|██▋       | 19/72 [00:21<00:52,  1.01it/s]C:\Users\allll\PycharmProjects\проект мага\.venv\Lib\site-packages\sklearn\base.py:1365: ConvergenceWarni


=== ЛУЧШАЯ КОНФИГУРАЦИЯ (по Calinski-Harabasz) ===
algo=agglo, K=10, z=2, run=1


TypeError: only length-1 arrays can be converted to Python scalars

python exe: C:\Users\allll\PycharmProjects\проект мага\.venv\Scripts\python.exe
python ver: 3.12.2 Windows
torch ver: 2.9.0+cpu
torch cuda build: None
cuda.is_available: False
cuda.device_count: 0
CUDA_VISIBLE_DEVICES: None
